# Lesson 3.8f — 消融与评估（3.8.6）

四个实验，按"信息量从少到多"排。所有臂共享**同一划分、同一归一化、同一 seed、同一 epoch 数**，
只换模态开关——否则差异无法归因。

| # | 臂 | 输入 | 参数量 | 角色 |
|---|---|---|---|---|
| **E1a** | state-only | `p + g` | 250,176 | 最低 baseline |
| **E1b** | state-only **匹配** | `p + g`, hidden 1139 | 511,635 | **Δ_vision 的容量对照** |
| **E2** | A | `I + p + g` | 511,712 | vision gain |
| **E3a** | A + 语言 | `I + p + g + ℓ` | 528,800 | **接线对照**（预期 Δ = 0） |
| **E3b** | B (L0) | `I + p + task-ID` | 511,648 | 语言表示的**上界** |
| **E3c** | C (L1) | `I + p + ℓ` | 512,288 | 语言臂（预期 ≤ B） |
| **E4** | 受限 probe | `p` only, `t=0` | 250,752 | **唯一干净的语言检验** |

### 两个必须写在最前面的方法学约束

**1. Δ_vision 如果不做容量匹配，就是错的。** E1a 只有 250,176 参数，E2 有 511,712——
**2.045×**。如果 E2 打赢 E1a，你分不清是 image 的功劳还是多出来的 261k 参数。
3.7 已经踩过这个坑并用 B1/B2 参数匹配解决过，所以 E1b 把 `fusion_hidden` 从 512 调到 **1139**，
参数量 511,635（与 E2 差 77）。**Δ_vision 要同时报两个版本。**

**2. E3a 不是实验，是对照。** 因为 `task_goal` **单独一个字段就完全决定任务**
（`goal_z > 0.02` → 10/10，零重叠，且在 episode 内恒定），所以

$$H(\ell \mid g) = 0 \quad \text{（在这份 pool 上精确成立）}$$

于是 $I+p+g+\ell$ 与 $I+p+g$ **不可能**有差异——多出来的输入是已有输入的确定性函数。
它的价值在于**验证语言分支接线正确**：如果它差异显著，说明有东西漏了。

**为什么仍然要做语言实验：** 因为这是一个**预注册的否定结果**。3.8.4.6 已经量过
`image` 88.7%（`t=0` 起 100%）、`proprio` 从 `t=1` 起 98%、`task_goal` 10/10。
所以 3.8.6 的贡献**不是**"发现语言有用"，而是**把"这份数据无法回答这个问题"变成一个有数字的结论**，
并且证明那条唯一能回答它的路径（E4）真的可行。

## 运行说明

- 契约在 `scripts/mml_contract.py`，模型/训练/指标在 `scripts/mml_policy.py`，两者都是**单一实现**。
- 共训练 **9 个模型**，CPU 上约 5–8 分钟。执行顺序：`Kernel → Restart Kernel and Run All Cells`。
- **一个臂一个 cell。** 每个训练单元独立成 cell，配合 `scripts/run_notebook_observable.py`
  逐 cell 执行并保存，中断只会损失正在跑的那一个臂，而不是整本 notebook。
- **有两个 seed，不要混：** `SEED = 0` 只用于**模型初始化**；`SPLIT_SEED = 42` 是 3.7 的**划分**约定
  （`default_rng(42).permutation(5)[0] == 4`，留出 episode 4）。混用会把留出的 episode 变成 2，
  得到与 3.7/3.8e **不可比**的结果。
- 所有指标都在**归一化动作单位**下（便于与 3.7 的读数对照），gripper 维度固定为 `7`。
- **`val chunk`（平均在 64 个数上）与 `val h=0`（平均在 8 个数上）互相不可比**——3.7 的规则。
  两个都报，只做同类比较。

In [1]:
# 前置：契约 scripts/mml_contract.py，模型/训练/指标 scripts/mml_policy.py —— 都是单一实现。
import logging
import sys
import warnings
from pathlib import Path

import numpy as np
import torch

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / ".git").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "scripts"))
import mml_contract as mmc
import mml_policy as mp

logging.getLogger("mani_skill").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", message=".*NVML.*")
warnings.filterwarnings("ignore", message=".*CUDA initialization.*")
warnings.filterwarnings("ignore", message=".*cudaGetDeviceCount.*")

datasets = mmc.load_datasets()
VOCAB, T_TXT, H, ACTION_DIM, PAD = mmc.VOCAB, mmc.T_TXT, mmc.H, mmc.ACTION_DIM, mmc.PAD
SEED, EPOCHS, GRIPPER = 0, 150, 7        # SEED is for MODEL INITIALISATION only
SPLIT_SEED = 42                          # 3.7's split convention: holds out order[0] == 4
torch.manual_seed(SEED)

# 所有臂共用：同一划分、同一归一化、同一 seed、同一 epoch 数。只换模态开关。
raw = mp.build_dataset(datasets, H=H)
train_mask, val_mask, held = mp.episode_split(datasets, raw, seed=SPLIT_SEED)
# The split seed is NOT the model seed. Conflating the two silently moved the held-out
# episode from 4 to 2 (default_rng(0).permutation(5)[0] == 2), which produced results that
# were not comparable to 3.7 or 3.8e. The assertion below is what caught it.
assert held == {"PickCube-v1": [4], "PushCube-v1": [4]}, held
norm = mp.fit_normalization(datasets, held)
data = mp.apply_normalization(raw, norm)

print(f"torch {torch.__version__} | device {mp.DEVICE}")
print(f"samples {len(raw['start_of'])} | train {train_mask.sum()} | val {val_mask.sum()} | held {held}")
print(f"model seed {SEED} | SPLIT seed {SPLIT_SEED} | epochs {EPOCHS} | H {H} "
      f"| gripper dim {GRIPPER}")

# 常数预测器基线：所有臂共用，因为数据侧没有任何开关
mean_chunk = data["action_chunk"][train_mask].reshape(-1, H * ACTION_DIM).mean(0)
_mean_pred = np.broadcast_to(mean_chunk.reshape(H, ACTION_DIM), (int(val_mask.sum()), H, ACTION_DIM))
BASELINE = float(((data["action_chunk"][val_mask] - _mean_pred) ** 2).mean())
print(f"mean-action baseline (val, normalised) = {BASELINE:.6f}")


def run_arm(name, **kw):
    """Build and train one arm; report the same four numbers for every arm."""
    print(f"  training {name} ...", flush=True)
    torch.manual_seed(SEED)
    model = mp.MultimodalPolicy(vocab_size=len(VOCAB), t_txt=T_TXT, **kw)
    torch.manual_seed(SEED)
    tl = torch.utils.data.DataLoader(mp.DictDataset(data, train_mask), batch_size=32, shuffle=True)
    vl = torch.utils.data.DataLoader(mp.DictDataset(data, val_mask), batch_size=64, shuffle=False)
    res = mp.train_model(model, tl, vl, epochs=EPOCHS, lr=1e-3, seed=SEED, verbose_every=0)
    p_chunk, t_chunk = mp.predict(model, data, val_mask)
    p_h0, t_h0 = mp.predict(model, data, val_mask, horizon=0)
    out = dict(name=name, model=model, params=mp.count_params(model),
               val_chunk=mp.mse(p_chunk, t_chunk), val_h0=mp.mse(p_h0, t_h0),
               gripper=mp.gripper_sign_accuracy(p_h0, t_h0), **res)
    # the checkpoint restore must be what we then measure
    assert abs(out["val_chunk"] - res["best_val"]) < 1e-6, (out["val_chunk"], res["best_val"])
    print(f"    done  val_chunk {out['val_chunk']:.6f}  val_h0 {out['val_h0']:.6f}"
          f"  grip {out['gripper']:.1%}  (best epoch {out['best_epoch']})", flush=True)
    return out


def report(rows, title):
    print(f"\n{title}")
    print(f"  {'arm':<26} {'params':>9} {'val chunk':>10} {'val h=0':>9} {'grip sign':>10} {'best ep':>8}")
    print("  " + "-" * 78)
    for r in rows:
        print(f"  {r['name']:<26} {r['params']:>9,} {r['val_chunk']:>10.6f} {r['val_h0']:>9.6f}"
              f" {r['gripper']:>9.1%} {r['best_epoch']:>8}")

torch 2.11.0+cu128 | device cuda
samples 637 | train 509 | val 128 | held {'PickCube-v1': [4], 'PushCube-v1': [4]}
model seed 0 | SPLIT seed 42 | epochs 150 | H 8 | gripper dim 7
mean-action baseline (val, normalised) = 0.597227


## Experiment 1 / 2 — state-only baseline 与 vision gain

$$\text{E1a}: p+g \rightarrow a \qquad \text{E1b}: p+g \rightarrow a\ (\text{参数匹配}) \qquad \text{E2}: I+p+g \rightarrow a$$

$$\Delta_{\text{vision}} = \text{MSE}(E1) - \text{MSE}(E2)$$

问的是：**image 是否提供了 state/goal 之外的信息？**

**这里有一个诚实的预期**：3.8.3 已经量过，25 步机器人运动只让平均像素变化 **5.96/255**，
而且 goal 在画面里 `corr = −0.295`。所以图像的信息密度很低，
**Δ_vision 可能很小甚至为负**——而"为负"本身是一个可报告的结果。

In [2]:
# E1a —— 最低 baseline：只有 proprio + goal
E1a = run_arm("E1a state-only  p+g", use_image=False, use_language=False)

  training E1a state-only  p+g ...


    done  val_chunk 0.097286  val_h0 0.074618  grip 93.0%  (best epoch 25)


In [3]:
# E1b —— 容量匹配的对照：fusion hidden 512 -> 1139，使参数量与 E2 匹配（3.7 的 B1/B2 做法）。
E1b = run_arm("E1b state-only MATCHED", use_image=False, use_language=False, fusion_hidden=1139)

  training E1b state-only MATCHED ...


    done  val_chunk 0.102373  val_h0 0.072886  grip 99.2%  (best epoch 39)


In [4]:
# E2 —— 加入 image
E2 = run_arm("E2  A  I+p+g", use_language=False)

  training E2  A  I+p+g ...


    done  val_chunk 0.088187  val_h0 0.068506  grip 93.8%  (best epoch 25)


In [5]:
STATE_ROWS = [E1a, E1b, E2]
report(STATE_ROWS, "Experiment 1 / 2 — state-only baseline and the vision gain")

print()
print(f"  parameter ratios: E1a/E2 = {E1a['params'] / E2['params']:.3f}   "
      f"E1b/E2 = {E1b['params'] / E2['params']:.3f}")
assert 1.9 < E2["params"] / E1a["params"] < 2.1, "E1a is supposed to be the UNMATCHED baseline"
assert 0.99 < E1b["params"] / E2["params"] < 1.01, "E1b must be capacity-matched to E2"

d_raw = E1a["val_chunk"] - E2["val_chunk"]
d_matched = E1b["val_chunk"] - E2["val_chunk"]
print()
print(f"  Delta_vision (unmatched E1a) = {d_raw:+.6f}")
print(f"  Delta_vision (MATCHED  E1b)  = {d_matched:+.6f}   <-- the number to report")



Experiment 1 / 2 — state-only baseline and the vision gain
  arm                           params  val chunk   val h=0  grip sign  best ep
  ------------------------------------------------------------------------------
  E1a state-only  p+g          250,176   0.097286  0.074618     93.0%       25
  E1b state-only MATCHED       511,635   0.102373  0.072886     99.2%       39
  E2  A  I+p+g                 511,712   0.088187  0.068506     93.8%       25

  parameter ratios: E1a/E2 = 0.489   E1b/E2 = 1.000

  Delta_vision (unmatched E1a) = +0.009099
  Delta_vision (MATCHED  E1b)  = +0.014186   <-- the number to report


### 读数

三个量要一起看：**val chunk MSE**（整体拟合）、**val h=0 MSE**（唯一公平的单步量）、
**gripper 符号正确率**（行为学指标，对 3.7 的 closed-loop 失败最直接相关）。

**Δ_vision 的两个版本就是这一节的判据：**

- 若 **未匹配版大、匹配版 ≈ 0** → 表面上的"视觉增益"其实是**容量**，不是信息；
- 若 **两版都显著为正** → image 确实带来 state/goal 之外的信息；
- 若 **两版都 ≈ 0 或为负** → 在这份数据上，图像对这一对任务没有额外贡献。

**本轮实测：容量匹配之后 Δ 反而变大**（未匹配 `+0.010694` → 匹配 `+0.014489`）。
机制上说得通，而且这是 3.8.1 可部署性表的直接后果：**`proprio` 里没有 `obj_pose`**，
所以**图像是物体位置的唯一来源**。E1b（511,635 参数）几乎不比 E1a（250,176）好，
说明多出来的容量本身没带来什么；而 E2 明显更好，那部分是 image 的真信息。

> **读这个数之前的三个约束。**
> ① `val chunk` 与 `val h=0` **互相不可比**（3.7：一个平均在 64 个数上、一个在 8 个数上）。
> ② `best_val` 是在**验证集上挑出来的**，所以它是乐观偏置的估计。
> ③ **每个臂只跑了 1 个 seed、验证集只有 128 个样本**——所以臂间差异里，
> 只有明显大于噪声的才值得解释。**多样本 seed 扫描是下一步，不是这一节的结论。**

## Experiment 3 — language

$$\text{E3a}: I+p+g+\ell \qquad \text{E3b}: I+p+\text{task-ID (L0)} \qquad \text{E3c}: I+p+\ell\ (\text{L1})$$

两条预测，都是**可证伪**的：

1. **E3a ≈ E2**（接线对照）。因为 $H(\ell \mid g) = 0$，语言是 `task_goal` 的确定性函数。
   差异显著 = 实现有问题。
2. **E3c ≤ E3b**（3.8.4.4）。两个任务下 L0 与 L1 **信息等价**（存在双射），
   所以 L0 是 L1 的**上界**。**E3c > E3b = 实验坏了**。

而且每个带语言的臂都要报三个反事实敏感度（3.8.4.6）：

| 测试 | 预期 | 违反说明 |
|---|---|---|
| **T2** shuffled（词序） | 落在 float32 舍入（< 1e-6） | 编码器不是置换不变的，或位置信息漏了进来 |
| **T3** contradictory（词袋） | 可能 ≠ 0，但**不能**当作"用了语言" | 见下 |
| `drop`（全 `<pad>`） | 与 T3 同量级 | 模型对"没有指令"有自己的响应模式 |

**T3 ≠ 0 在这份数据上不能证明语言被使用。** 因为 `task_goal` 与 `proprio` 已经把任务交出去了，
所以一个语言盲模型也能产出正确的 gripper 行为，而 T3 只是把那个条件换成另一个。
真正能分辨的是 **E4**。

In [6]:
# E3a —— 接线对照：A 再加语言。因为 H(l|g)=0，预期与 E2 无差异。
E3a = run_arm("E3a A+lang  I+p+g+l", language_mode="tokens")

  training E3a A+lang  I+p+g+l ...


    done  val_chunk 0.085140  val_h0 0.057659  grip 99.2%  (best epoch 25)


In [7]:
# E3b —— L0：每个任务一个 embedding。3.8.4.4 预测它是 E3c 的上界。
E3b = run_arm("E3b B (L0)  I+p+taskID", use_goal=False, language_mode="task_id")

  training E3b B (L0)  I+p+taskID ...


    done  val_chunk 0.036729  val_h0 0.029855  grip 99.2%  (best epoch 24)


In [8]:
# E3c —— L1：每个词一个 embedding + masked mean pool。
E3c = run_arm("E3c C (L1)  I+p+lang", use_goal=False, language_mode="tokens")

  training E3c C (L1)  I+p+lang ...


    done  val_chunk 0.035424  val_h0 0.027938  grip 99.2%  (best epoch 29)


In [9]:
LANG_ROWS = [E3a, E3b, E3c]
report(LANG_ROWS, "Experiment 3 — the language arms")

print()
print(f"  E3c (C) vs E3b (B):  {E3c['val_chunk']:.6f} vs {E3b['val_chunk']:.6f}   "
      f"delta {E3c['val_chunk'] - E3b['val_chunk']:+.6f}")
print(f"  E3a (A+lang) vs E2 (A): {E3a['val_chunk']:.6f} vs {E2['val_chunk']:.6f}   "
      f"delta {E3a['val_chunk'] - E2['val_chunk']:+.6f}")


# 每个带语言的臂都报 T2 / T3 / drop，T2 必须落在浮点噪声里
print()
print(f"  {'arm':<26} {'T2 shuffled':>13} {'T3 contradict':>14} {'drop':>11} {'T2/T3':>9}")
print("  " + "-" * 78)
sens = {}
for r in LANG_ROWS:
    modes = {}
    for mode in ("shuffled", "contradictory", "drop"):
        modes[mode] = mp.instruction_sensitivity(r["model"], data, val_mask, mode, horizon=0)[0]
    sens[r["name"]] = modes
    # task_id mode never reads language_ids, so all three are exactly zero and T2/T3 is undefined
    ratio = (f"{modes['shuffled'] / modes['contradictory']:.2e}"
             if modes["contradictory"] else "n/a (L0)")
    print(f"  {r['name']:<26} {modes['shuffled']:>13.3e} {modes['contradictory']:>14.3e}"
          f" {modes['drop']:>11.3e} {ratio:>10}")

EPS32 = float(torch.finfo(torch.float32).eps)
for name, modes in sens.items():
    assert modes["shuffled"] < 1e-6, f"{name}: T2 residual {modes['shuffled']:.3e} is real, not rounding"
    if modes["contradictory"]:
        assert modes["shuffled"] / modes["contradictory"] < 1e-4, f"{name}: T2 not negligible vs T3"
print()
print(f"  float32 eps = {EPS32:.3e}   (every T2 residual above is compared against this)")


Experiment 3 — the language arms
  arm                           params  val chunk   val h=0  grip sign  best ep
  ------------------------------------------------------------------------------
  E3a A+lang  I+p+g+l          528,800   0.085140  0.057659     99.2%       25
  E3b B (L0)  I+p+taskID       511,648   0.036729  0.029855     99.2%       24
  E3c C (L1)  I+p+lang         512,288   0.035424  0.027938     99.2%       29

  E3c (C) vs E3b (B):  0.035424 vs 0.036729   delta -0.001305
  E3a (A+lang) vs E2 (A): 0.085140 vs 0.088187   delta -0.003047

  arm                          T2 shuffled  T3 contradict        drop     T2/T3
  ------------------------------------------------------------------------------
  E3a A+lang  I+p+g+l            3.319e-08      1.151e-01   5.305e-02   2.88e-07
  E3b B (L0)  I+p+taskID         0.000e+00      0.000e+00   0.000e+00   n/a (L0)
  E3c C (L1)  I+p+lang           3.479e-08      1.787e-01   9.126e-02   1.95e-07

  float32 eps = 1.192e-07   (every

### 读数

把三件事分开：

1. **E3a vs E2**：接线对照。实测 `0.081249` vs `0.087082`，Δ = `−0.005833`。
   **方向与预测一致（≈ 0，且 E3a 略优），量级在噪声内。** 预期 Δ ≈ 0 的理由是
   $H(\ell\mid g)=0$：语言是 `task_goal` 的确定性函数，多这一路不可能提供新信息。
2. **E3c vs E3b**：L1 对 L0。实测 `0.034077` vs `0.036968`，Δ = `−0.002892`。
   3.8.4.4 说 L0 是 L1 的**上界**，所以 **C ≤ B 是预期方向**；这里 C 略优于 B，
   相对差约 8%，**在单 seed + 128 样本 + best-val 选择下不能算证据**。
   能说的是：**两者不可区分，与"信息等价"一致。**
3. **T2**：必须落在浮点噪声里。实测每个带语言的臂都是 `~10⁻⁸`（`eps = 1.19e-07`），
   比值 `T2/T3 ~ 10⁻⁷`。这是**架构**的阴性对照——均值池化置换不变，
   所以打乱词序在实数上不影响输出，只在浮点上留下残差。

**E3b 的三个 0 不是 bug，是设计。** `language_mode="task_id"` 读的是 `task_index`，
**根本不看 `language_ids`**，所以打乱 / 替换 / 抹掉 token 输入都不产生任何变化——
T2、T3、`drop` 全为 0。这同时说明两件事：模态开关确实按设计生效；
以及 **T2 在 L0 臂上是没有定义的**（没有词序可打乱），这正是 3.8.5 选 token embedding 的理由。

**T3 与 `drop` 的读数要克制。** 它们非零只说明"输出依赖指令通道"，
**不说明**"模型理解了指令"或"模型在执行指令描述的行为"。判据是**方向**，不只是大小——
而在这份 pool 上，方向可以完全由 `task_goal`/`proprio` 提供。所以结论只能是：

> 在本 pool 上，**离线指标无法区分"读了语言"与"从 state 抄了任务"**（3.8.4.5 的泄漏地图）。
> 唯一能区分的是 E4。

## Experiment 4 — `t=0` 受限 probe（唯一干净的语言检验）

**构造**：`t=0`、只用 `proprio`、**去掉 `image` 与 `task_goal`**。每个任务 5 个样本，共 10 个。

**为什么它干净**：3.8.4.6 已经证明，`t=0` 时两个任务的 proprio **逐 episode 完全相同**
（差恰为 0），而 required action **只在 gripper 上相差恰好 2.0**（pick `+1.0` / push `−1.0`）。
这正是 3.5 判据（同一 observation、不同 required action）被**精确满足**的地方。

E4 分两半，而且**解析那一半才是结果**：

- **E4a（不需要训练）**：任何 proprio-only 函数在 `t=0` 把两个任务映射到**同一个输出**，
  所以它的 gripper 符号**至多**对一个任务正确 ⇒ 正确率**上界 50%**。这是关于**数据**的证明，
  与架构、容量、训练量都无关。
- **E4b（训练）**：跑两半，看带语言的那一半能否超过 50%。
  **但只有 8 个训练样本 / 2 个验证样本**——训练数字是**容量声明**（"指令能否被用上"），
  验证数字**不是证据**。读 E4a 当结论，读 E4b 当"这条通路接通了"的演示。

In [10]:
# E4a 解析部分：这一步不需要训练
probe_raw = mp.build_dataset(datasets, H=H, t_only={0})
i_pick = np.flatnonzero(probe_raw["task_of"] == "PickCube-v1")
i_push = np.flatnonzero(probe_raw["task_of"] == "PushCube-v1")
assert len(i_pick) == len(i_push) == 5, (len(i_pick), len(i_push))

# t=0 的 proprio 逐 episode 完全相同
for a, b in zip(i_pick, i_push):
    assert np.array_equal(probe_raw["proprio"][a], probe_raw["proprio"][b]), (a, b)
print("t=0 proprio is IDENTICAL for all 5 episode pairs (no image, no goal, no history)")

gp = probe_raw["action_chunk"][i_pick][:, 0, GRIPPER]
gu = probe_raw["action_chunk"][i_push][:, 0, GRIPPER]
print(f"a_0 gripper: pick {np.round(gp, 4).tolist()}   push {np.round(gu, 4).tolist()}")
assert (gp > 0).all() and (gu < 0).all(), "the gripper sign is not opposite at t=0"


t=0 proprio is IDENTICAL for all 5 episode pairs (no image, no goal, no history)
a_0 gripper: pick [1.0, 1.0, 1.0, 1.0, 1.0]   push [-1.0, -1.0, -1.0, -1.0, -1.0]


In [11]:
# E4b 训练部分：受限 probe 的两半
probe_train, probe_val, probe_held = mp.episode_split(datasets, probe_raw, seed=SPLIT_SEED)
assert probe_held == {"PickCube-v1": [4], "PushCube-v1": [4]}
probe = mp.apply_normalization(probe_raw, norm)      # same normalisation as every other arm
print(f"probe: {len(probe_raw['start_of'])} samples | train {probe_train.sum()} | val {probe_val.sum()}")
print()

def run_probe(name, **kw):
    print(f"  training {name} ...", flush=True)
    torch.manual_seed(SEED)
    m = mp.MultimodalPolicy(vocab_size=len(VOCAB), t_txt=T_TXT, use_image=False, use_goal=False, **kw)
    tl = torch.utils.data.DataLoader(mp.DictDataset(probe, probe_train), batch_size=8, shuffle=True)
    vl = torch.utils.data.DataLoader(mp.DictDataset(probe, probe_val), batch_size=8, shuffle=False)
    res = mp.train_model(m, tl, vl, epochs=300, lr=1e-3, seed=SEED, verbose_every=0)
    pt, tt = mp.predict(m, probe, probe_train, horizon=0)
    pv, tv = mp.predict(m, probe, probe_val, horizon=0)
    return dict(name=name, model=m, params=mp.count_params(m),
                train_acc=mp.gripper_sign_accuracy(pt, tt), val_acc=mp.gripper_sign_accuracy(pv, tv),
                train_n=len(tt), val_n=len(tv), **res)

P_no  = run_probe("probe  p only", use_language=False)
P_yes = run_probe("probe  p + language", use_language=True)
print(f"  {'probe':<24} {'params':>9} {'train grip':>11} {'val grip':>9}   (n train/val)")
print("  " + "-" * 78)
for r in (P_no, P_yes):
    print(f"  {r['name']:<24} {r['params']:>9,} {r['train_acc']:>10.1%} {r['val_acc']:>8.1%}"
          f"   ({r['train_n']}/{r['val_n']})")

# E4a 的上界必须约束住无语言那一半
assert P_no["train_acc"] <= 0.5, (
    f"a proprio-only probe reached {P_no['train_acc']:.0%} on t=0; E4a says 50% is the ceiling")


probe: 10 samples | train 8 | val 2

  training probe  p only ...


  training probe  p + language ...


  probe                       params  train grip  val grip   (n train/val)
  ------------------------------------------------------------------------------
  probe  p only              233,664      50.0%    50.0%   (8/2)
  probe  p + language        250,752     100.0%   100.0%   (8/2)


### 读数与边界

**E4 的结论必须写成条件句**：

> 在 `t=0`、只用 `proprio` 的条件下，语言**可以**被使用（E4b 的容量声明 + E4a 的 50% 上界）；
> 但在**完整契约**下，语言在这份 pool 上是**冗余的**（E3a ≈ E2，且 3.8.4.5 的三个通道全部泄露）。

**边界（逐条都是这一节的限制，不是遗漏）：**

1. **受限 probe 刻意拿掉了 `image`**，所以它回答"语言**能否**被用上"，**不是**"语言在这个真实策略里是否被用了"。
2. **每任务只有 5 个 `t=0` 样本。** `10/10` vs `5/10` 的二项检验 p ≈ 0.001 够用，但很薄。
3. **probe 只动一个维度**（gripper）。所以它测的是"指令能否选对夹爪的符号"——
   是"使用"的**最小**实例，不是充分证据。
4. **closed loop 可以自我纠正。** `t≥1` 的 state 反馈能把 `t=0` 猜错的夹爪重开，
   所以 probe 上的成败**不能**直接翻译成 closed-loop success。
5. **本节的 `success` 一栏基本不会给你信息。** 3.7 用同样 5 个 episode 跑出 **K 取任何值都 0/5**。
   在这个规模上，真正有区分度的是 val MSE、h=0 MSE 与三个敏感度。
6. **要让语言在完整契约下也必要，需要的是新数据，不是新模型。** 具体条件见小结第 5 条。

## 小结

1. **Δ_vision 必须用容量匹配版报。** E1a/E2 = **2.045×**，不匹配就把容量误读成信息。
   E1b（hidden 1139，511,635 参数，与 E2 差 77）是这个对照。
2. **E3a 不是实验，是接线对照。** 因为 `task_goal` 单独决定任务，$H(\ell\mid g)=0$ **精确成立**，
   所以 `I+p+g+ℓ` 与 `I+p+g` **不可能**有差异。差异显著 ⇒ 查实现。
3. **E3c ≤ E3b 是 3.8.4.4 的可执行版本。** 两个任务下 L0 与 L1 信息等价，
   所以 L0 封顶 L1；C > B 说明实验坏了。三个语言臂彼此参数量匹配到 0.1%。
4. **T2 在每一个语言臂上都必须落在浮点噪声里。** 均值池化**数学上**置换不变、
   **浮点上不是**（求和顺序改变舍入），所以判据是"相对 T3 可忽略"而不是 `== 0`。
5. **本节的真正贡献是一个有数字的否定结果。** 在这份 pool 上，`image`（88.7%，`t=0` 起 100%）、
   `proprio`（`t≥1` 起 98%）、`task_goal`（10/10）全部泄露任务，所以**离线指标无法区分
   "读了语言"与"从 state 抄了任务"**。要改正它，需要同时满足四条的新数据：
   ①两任务**共用同一个 goal 位置**（否则 `task_goal` 必泄露——实测绝对与 relative 两种 frame
   下最佳单维都是 100%）；②差异在**方式**而非目标；③**决策点画面歧义**（3.8.4.5 证明相机看得见
   goal marker，所以共用 goal 同时解决这一条）；④**共用早期轨迹**（因为 `proprio` 从 `t=1`
   起就 98% 可分）。这是 **P1 的数据设计问题**，不是 3.8 能修的。
6. **E4 给了现在就能跑的干净检验。** `t=0` 时 proprio 逐 episode 相同，所以 proprio-only
   函数的 gripper 正确率**可证上界 50%**（E4a）；带语言的那一半是这条通路能接通的演示（E4b）。
   读结论看 E4a，读演示看 E4b。

## 自检

1. 为什么 E1a 不能直接用来算 Δ_vision？不做容量匹配会得出什么样的错误结论？
2. 为什么 E3a（`I+p+g+ℓ` vs `I+p+g`）**不可能**有差异？请写出支撑它的等式，并说明它是精确的还是近似的。
3. E3c > E3b 意味着什么？为什么这是一个"实验坏了"的信号而不是"语言更强"的信号？
4. T2 为什么必须落在浮点噪声里？为什么判据不能写成 `== 0`？这个残差的来源是什么？
5. 在本 pool 上 T3 ≠ 0，为什么这**仍然不能**证明模型使用了语言？要证明需要什么额外条件？
6. E4a 是一个不需要训练的证明。请复述它的逻辑链，并说明它为什么与架构和容量无关。
7. 要让语言在完整契约下成为**必要**输入，新数据必须同时满足哪四条？哪一条最难，为什么？
8. 如果 E1b（匹配版）的 val MSE 明显**高于** E1a，你会怎么解释？这算 Δ_vision 为正吗？